# Stage 4: held-out evaluation and a live API demo

Compares three systems on positions no stage has touched: the plain instruct model (baseline), the SFT adapter, and the DPO adapter. Every answer goes through the same deterministic verifier, so the factual numbers are engine-checked counts, not opinions. Then the FastAPI service runs here in the notebook against the DPO model, as proof the serving path works with real weights.

- **Runtime type**: **T4 GPU**.
- **Time**: about 30 to 60 minutes.


In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU, then rerun."
print(torch.cuda.get_device_name(0))

In [ ]:
REPO_URL = "https://github.com/ariyan-tipu/opening-theory-engine.git"

# A plain install rather than -e: editable installs only register when the
# interpreter starts, and a live Colab kernel has already started.
!rm -rf /content/opening-theory-engine
!git clone --depth 1 {REPO_URL} /content/opening-theory-engine
%pip install -q "/content/opening-theory-engine[train,gpu]"
print("install done")

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
%cd /content/drive/MyDrive/opening-theory-engine

CONFIG = "/content/opening-theory-engine/configs/colab-t4.yaml"

In [ ]:
# The verifier needs the engine for the serving demo later; fetch it now.
import os

from opening_theory_engine.engine.fetch import fetch

os.environ["STOCKFISH_PATH"] = str(fetch())

In [ ]:
# The evaluation: baseline vs SFT vs DPO on the held-out split.
!ote eval run --config {CONFIG}

In [ ]:
print(open("results/summary.md", encoding="utf-8").read())

In [ ]:
# Live serving demo: the real API process, the DPO adapter, the real engine.
import os
import subprocess
import time

env = dict(
    os.environ,
    OTE_BACKEND="hf",
    OTE_BASE_ID="Qwen/Qwen2.5-3B-Instruct",
    OTE_ADAPTER_PATH="artefacts/dpo/adapter",
    OTE_FOUR_BIT="true",
    OTE_ENGINE="stockfish",
)
server = subprocess.Popen(
    ["ote", "serve", "--port", "8000"], env=env,
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)

import httpx

for _ in range(120):  # the model takes a while to load the first time
    try:
        if httpx.get("http://127.0.0.1:8000/health", timeout=2).status_code == 200:
            break
    except httpx.HTTPError:
        time.sleep(5)
print(httpx.get("http://127.0.0.1:8000/health", timeout=10).json())

In [ ]:
import json

import httpx

response = httpx.post(
    "http://127.0.0.1:8000/explain",
    json={
        "fen": "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1",
        "move": "e4",
        "opening": "King's Pawn Game",
        "eco": "B00",
    },
    timeout=300,
)
print(json.dumps(response.json(), indent=2))
server.terminate()

## Bring the results home

Copy these files from your Drive into your local clone and commit them, so the README numbers stay backed by artefacts in the repository:

- `results/summary.md` and `results/summary.json` (straight copies)
- `data/processed/dataset_card.md`, committed as `results/dataset_card.md`

The README results section explains the numbers; `docs/results.md` is where the narrative write-up lives.
